# Stage-Based Modeling Walkthrough

This notebook explains and runs the Gate 0, Gate 1, Gate 2, and Axis B modeling scripts. The scripts remain the canonical source for reproduced outputs; this notebook is a readable companion.

## 1. Load data and inspect gate samples

In [1]:
from pathlib import Path
import sys

# Resolve repository root even when Jupyter starts in a different working directory.
ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "analysis").exists() and (candidate / "dataset").exists():
        ROOT = candidate
        break
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
from analysis.common import load_analysis_dataset

df = load_analysis_dataset(ROOT)
df.head()

,Case ID,PR_Link,Conversation_Link,Outcome_Class,Context,Specificity,Verification,Rationale,PQS,PR_Size,...,PQS,Repository,PR_Number,Merged,Closed,Generated_Code,Adopted_Code,Resolved,Close_Event,Merge_Event
0,PA-1,https://github.com/Altinn/altinn-broker/pull/259,https://chat.openai.com/share/b7853f70-84b8-47...,PA,1.0,1.0,0.0,Context was scored 1 because the prompt provid...,2,125.0,...,2,Altinn/altinn-broker,259,1,0,1,1,1,0,1
1,PA-2,https://github.com/Hochfrequenz/kohlrahbi/pull...,https://chat.openai.com/share/4ad4c1ad-6f13-4a...,PA,1.0,1.0,0.0,Context was scored 1 because the prompt provid...,2,51.0,...,2,Hochfrequenz/kohlrahbi,158,1,0,1,1,1,0,1
2,PA-3,https://github.com/MartinsOnuoha/what-should-i...,https://chat.openai.com/share/2aa6268a-7a4e-47...,PA,2.0,2.0,2.0,Context was scored 2 because the prompt mentio...,6,300.0,...,6,MartinsOnuoha/what-should-i-design,8,1,0,1,1,1,0,1
3,PA-4,https://github.com/Opetushallitus/ludos/pull/102,https://chat.openai.com/share/bdfcb857-08a3-4f...,PA,1.0,1.0,1.0,Context was scored 1 because the prompt provid...,3,620.0,...,3,Opetushallitus/ludos,102,1,0,1,1,1,0,1
4,PA-5,https://github.com/SharezoneApp/sharezone-app/...,https://chat.openai.com/share/fd82b66d-d949-43...,PA,2.0,1.0,1.0,Context was scored 2 because the prompt mentio...,4,18.0,...,4,SharezoneApp/sharezone-app,980,1,0,1,1,1,0,1


In [ ]:
gate0 = df[df.Outcome_Class.isin(['PA','PN','NE'])]
gate1 = df[df.Outcome_Class.isin(['PA','PN'])]
gate2 = df[df.Outcome_Class.eq('PA')]
axisb = df.dropna(subset=['Time_To_Event'])
pd.DataFrame({'stage':['Gate 0','Gate 1','Gate 2','Axis B'], 'n':[len(gate0), len(gate1), len(gate2), len(axisb)]})

## 2. Gate 0: code generation

Gate 0 models whether a prompt produces actionable code, contrasting NE against PA/PN cases.

In [ ]:
from analysis.quantitative import gate0_generation
gate0_results = gate0_generation.run(ROOT)
gate0_results

## 3. Gate 1: code adoption

Gate 1 is restricted to PA/PN cases and models whether generated code was adopted.

In [ ]:
from analysis.quantitative import gate1_adoption
gate1_results = gate1_adoption.run(ROOT)
gate1_results

## 4. Gate 2: integration depth

Gate 2 is restricted to PA cases and models the fraction of generated code retained in the final implementation.

In [ ]:
from analysis.quantitative import gate2_integration
gate2_results = gate2_integration.run(ROOT)
gate2_results

## 5. Axis B: lifecycle outcomes

Axis B keeps pull-request lifecycle outcomes separate from prompt-level code generation/adoption/integration effects.

In [ ]:
from analysis.quantitative import axisB_lifecycle
axisb_results = axisB_lifecycle.run(ROOT)
axisb_results